<a href="https://colab.research.google.com/github/smduarte/spbd-2526/blob/main/docs/labs/lab1/SPBD_Labs_mapreduce1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reference **Pure** Python WordCount Example

Read the words from an input file and count them after removing punctuation.

In [ ]:
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

In [ ]:
# Simple word count in Python
import string

with open('os_maias.txt', 'r') as f:
    text = f.read()
    text = text.strip()
    text = text.translate(str.maketrans('', '', string.punctuation+'«»')).lower()
    word_count = len(text.split())
    print(f"Total words: {word_count}")

# MrJob MapReduce Python Example

Word count implemented in pure Python, using the library MrJob.

[MrJob](https://mrjob.readthedocs.io/en/latest/) can be used to write MapReduce jobs and run them on several platforms.

Some key advantages:
+ Write **multi-step** MapReduce jobs in pure Python;
+ Test on your **local machine**;
+ Deploy jobs in several cloud plataforms of several vendors.

In [ ]:
#@title Download the dataset and install MrJob
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

!pip install mrjob --quiet
!wget -q -O /etc/mrjob.conf https://raw.githubusercontent.com/smduarte/spbd-2526/main/docs/labs/lab1/mrjob.conf

# MrJob WordCount Example
Read the words from input and count them.

The processing is split into two main phases:

+ The mapper emits for each line the number of words
+ The reduces sums all the tuples produced by the mapper stage...

Using MrJob, a MapReduce job can be expressed in a single Python class,
with methods for each of the phases. The reducer phase is called separately for each key, with the collection of values to be reduced.

In [ ]:
%%file wordcount.py

import string
from mrjob.job import MRJob

class MRWordCount(MRJob):

    def mapper(self, _, line):
      # remove leading and trailing whitespace
      line = line.strip()
      # remove punctuation characters
      line = line.translate(str.maketrans('', '', string.punctuation+'«»')).lower()
      # split the line into words
      yield None, len(line.split())

    def reducer(self, _, values):
        yield "Total words:", sum(values)

if __name__ == '__main__':
    MRWordCount.run()

## Local Execution of MrJob programs

The output will be placed in a folder "results" that needs to be empty. The output is a set of files, each corresponding to a partition, produced by the

In [ ]:
!rm -rf results
!python3 -m wordcount -r local os_maias.txt --output-dir results --cleanup NONE

## Supplying a combiner...


In [ ]:
%%file wordcount2.py

import string
from mrjob.job import MRJob

class MRWordCount2(MRJob):

    def mapper(self, _, line):
      # remove leading and trailing whitespace
      line = line.strip()
      # remove punctuation characters
      line = line.translate(str.maketrans('', '', string.punctuation+'«»')).lower()
      # split the line into words
      yield None, len(line.split())

    def combiner(self, key, values):
        yield key, sum(values)

    def reducer(self, _, values):
        yield "Total words:", sum(values)

if __name__ == '__main__':
    MRWordCount2.run()

In [ ]:
!rm -rf results
!python3 -m wordcount2 -r local os_maias.txt --output-dir results --cleanup NONE